In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import time
import re
import json
import unicodedata
from datetime import datetime

# -----------------------
# Config
# -----------------------
BASE_URL = "https://www.salon.com/category/news-and-politics"
PAGE_URL_TEMPLATE = "https://www.salon.com/category/news-and-politics?pagenum={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Salon-Scraper/1.0)"
}

REQUEST_DELAY = 1.5
MAX_LINKS = 200
MIN_WORD_COUNT = 130

# -----------------------
# Utilities
# -----------------------
def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


def compute_article_id(url):
    slug = urlparse(url).path.rstrip("/").split("/")[-1]
    return slug.replace("-", "_")


def utc_now_iso():
    return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

# -----------------------
# Pagination
# -----------------------
def extract_article_links(html):
    soup = BeautifulSoup(html, "html.parser")
    links = set()

    for article_div in soup.find_all("div", class_="grid-article"):
        a_tag = article_div.find("a", href=True)
        if a_tag:
            href = a_tag["href"]
            if (
                href.startswith("https://www.salon.com/")
                and re.search(r"/20\d{2}/\d{2}/\d{2}/", href)
            ):
                links.add(href.split("?")[0])

    return list(links)


def crawl_all_pages(max_links=100):
    page = 1
    all_links = set()

    print(f"[start] Beginning pagination crawl (target: {max_links} articles)\n")

    while True:
        url = BASE_URL if page == 1 else PAGE_URL_TEMPLATE.format(page)
        print(f"[pagination] Fetching page {page}: {url}")

        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code != 200:
                print(f"[pagination] Stopping — HTTP {r.status_code}")
                break
        except Exception as e:
            print(f"[pagination] Error fetching page {page}: {e}")
            break

        links = extract_article_links(r.text)
        print(f"[pagination] Found {len(links)} article links on page {page}")

        if not links:
            print("[pagination] No links found — stopping crawl")
            break

        for link in links:
            if link not in all_links:
                all_links.add(link)
                print(f"  [+] Collected ({len(all_links)}/{max_links}): {link}")

            if len(all_links) >= max_links:
                print(f"\n[pagination] Reached target of {max_links} links\n")
                return list(all_links)

        page += 1
        time.sleep(REQUEST_DELAY)

    print(f"\n[pagination] Finished with {len(all_links)} total links\n")
    return sorted(all_links)


# -----------------------
# Text normalization
# -----------------------
def normalize_unicode(text):
    return unicodedata.normalize("NFKC", text)

def fix_encoding_issues(text):
    """Fix UTF-8 smart punctuation mis-decoded as Windows-1252"""

    text = (
        text
        .replace("â\u0080\u0098", "'")
        .replace("â\u0080\u0099", "'")
        .replace("â\u0080\u009c", '"')
        .replace("â\u0080\u009d", '"')
        .replace("â\u0080\u0093", "–")
        .replace("â\u0080\u0094", "—")
        .replace("â\u0080\u00a6", "...")
    )

    text = re.sub(r"[\u0080-\u009f]", "", text)
    return text


def normalize_quotes_and_apostrophes(text):
    return (
        text
        .replace("\u201c", '"')
        .replace("\u201d", '"')
        .replace("\u2018", "'")
        .replace("\u2019", "'")
        .replace("\u00ab", '"')
        .replace("\u00bb", '"')
        .replace("`", "'")
    )


def normalize_body_text(text):
    text = fix_encoding_issues(text)
    text = normalize_quotes_and_apostrophes(text)
    text = normalize_unicode(text)

    # Remove bracket artifacts
    text = re.sub(r"\[\s*\.\.\.\s*\]", " ", text)
    text = re.sub(r"\[([^\]]+)\]", r"\1", text)
    text = re.sub(r"\((cough|laughter|applause|pause)\)", " ", text, flags=re.I)

    # Normalize ellipsis + dashes
    text = text.replace("…", "...")
    text = text.replace("–", "—")
    text = re.sub(r"\s*—\s*", " — ", text)

    # Fix punctuation spacing
    text = re.sub(r"([.!?])(\w)", r"\1 \2", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)

    # Remove invisible characters
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)
    text = text.replace("\u00A0", " ")

    # Fix apostrophe joining issues inside words
    text = re.sub(r"(\w)'(\w)", r"\1'\2", text)

    return clean_text(text)

def should_skip_paragraph(txt):
    t = txt.lower()
    return (
        "start your day with essential news" in t
        or "sign up for our free" in t
        or "we need your help to stay independent" in t
        or "subscribe today to support salon" in t
    )


# -----------------------
# Article extractor
# -----------------------
def extract_article(url, index=None, total=None):
    label = f"[article {index}/{total}]" if index and total else "[article]"
    crawl_timestamp = utc_now_iso()

    print(f"{label} Fetching: {url}")
    print(f"{label} Crawl timestamp: {crawl_timestamp}")

    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        if r.status_code != 200:
            print(f"{label} Skipped — HTTP {r.status_code}")
            return None
    except Exception as e:
        print(f"{label} Error fetching article: {e}")
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    h1 = soup.find("h1")
    if not h1:
        print(f"{label} Skipped — no headline found")
        return None

    header = normalize_body_text(h1.get_text())

    time_tag = soup.find("time", {"datetime": True})
    published_date = time_tag["datetime"] if time_tag else None

    article_body = soup.find("section", class_="article__content")
    if not article_body:
        print(f"{label} Skipped — no article body found")
        return None

    for div in article_body.find_all("div", class_="layout_template_wrapper"):
        div.decompose()

    for el in article_body.find_all(["aside", "nav", "figcaption"]):
        el.decompose()

    for div in article_body.find_all("div", class_=re.compile(r"template|promo|ad|sponsor", re.I)):
        div.decompose()
    
    paragraphs = []
    for p in article_body.find_all("p"):
        txt = p.get_text(" ", strip=True)
        if not txt or should_skip_paragraph(txt):
            continue
        paragraphs.append(txt)

    body_text = normalize_body_text("\n\n".join(paragraphs))
    word_count = len(body_text.split())

    if word_count < MIN_WORD_COUNT:
        print(f"{label} Skipped — only {word_count} words (< {MIN_WORD_COUNT})")
        return None

    print(f"{label} ✓ Extracted ({word_count} words): {header[:70]}...")

    return {
        "article_id": compute_article_id(url),
        "header": header,
        "body_text": body_text,
        "word_count": word_count,
        "outlet": "Salon",
        "url": url,
        "published_date": published_date,
        "crawled_at": crawl_timestamp,
        "label": "left",
    }

# -----------------------
# Main
# -----------------------
if __name__ == "__main__":
    print("\n=== Salon Scraper Starting ===\n")

    links = crawl_all_pages(max_links=MAX_LINKS)
    print(f"[main] Collected {len(links)} article URLs\n")

    articles = []

    for i, link in enumerate(links, 1):
        article = extract_article(link, index=i, total=len(links))
        if article:
            articles.append(article)
        time.sleep(REQUEST_DELAY)

    print(f"\n[main] Extraction complete")
    print(f"[main] Successfully extracted {len(articles)} articles")
    print(f"[main] Skipped {len(links) - len(articles)} articles\n")

    if articles:
        output_file = "salon_articles.json"
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(articles, f, indent=2, ensure_ascii=False)

        print(f"[main] Saved {len(articles)} articles to {output_file}")

    print("\n=== Done ===\n")


=== Salon Scraper Starting ===

[start] Beginning pagination crawl (target: 200 articles)

[pagination] Fetching page 1: https://www.salon.com/category/news-and-politics
[pagination] Found 24 article links on page 1
  [+] Collected (1/200): https://www.salon.com/2026/04/07/will-you-stand-for-western-civilization-vance-demands-hungary-reelect-viktor-orban/
  [+] Collected (2/200): https://www.salon.com/2026/04/10/the-christian-rights-victim-complex-fuels-trumps-iran-war/
  [+] Collected (3/200): https://www.salon.com/2026/04/09/completely-false-melania-trump-denies-any-relationship-to-jeffrey-epstein/
  [+] Collected (4/200): https://www.salon.com/2026/04/07/trump-revealed-his-objective-in-iran-40-years-ago/
  [+] Collected (5/200): https://www.salon.com/2026/04/10/fragile-iran-war-ceasefire-holds-as-americans-feel-impacts-of-the-war/
  [+] Collected (6/200): https://www.salon.com/2026/04/09/reasonable-discussion-pentagon-denies-reports-that-official-threatened-pope-leo/
  [+] Collecte